In [ ]:
import glob, os, zipfile

MODEL_PATHS = []
for cfg in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
    d = os.path.dirname(cfg)
    out = f"/kaggle/working/{os.path.basename(d)}.keras"
    with zipfile.ZipFile(out, "w") as z:
        for f in os.listdir(d):
            z.write(os.path.join(d, f), f)
    MODEL_PATHS.append(out)

print(MODEL_PATHS)

In [ ]:
!pip install -q gradio

import numpy as np
import tensorflow as tf
import gradio as gr

# Daftar kelas (urutan harus sama dengan saat training)
CLASS_NAMES = ['Acne', 'Atopic Dermatitis', 'Eczema Herpeticum',
               'Keratosis', 'Psoriasis', 'Urticaria']
MODEL_PATHS = ['/kaggle/working/model_seed42.keras',
               '/kaggle/working/model_seed59.keras']

# Muat kedua model untuk ensemble
models = [tf.keras.models.load_model(p, compile=False) for p in MODEL_PATHS]


def predict(img):
    # Ubah ukuran gambar ke 384x384 dan tambahkan dimensi batch
    x = tf.image.resize(np.array(img.convert("RGB"), dtype="float32"), (384, 384))[None]

    # TTA: gambar asli, flip horizontal, flip vertikal, flip keduanya
    views = [x, tf.image.flip_left_right(x), tf.image.flip_up_down(x),
             tf.image.flip_up_down(tf.image.flip_left_right(x))]

    # Prediksi tiap model pada tiap view, ubah ke probabilitas (softmax)
    probs = [tf.nn.softmax(m.predict(v, verbose=0))[0].numpy()
             for m in models for v in views]

    # Rata-rata semua prediksi -> probabilitas akhir per kelas
    return dict(zip(CLASS_NAMES, np.mean(probs, axis=0).tolist()))


# Antarmuka web: upload gambar -> tampilkan 2 prediksi teratas
gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload foto kulit"),
    outputs=gr.Label(num_top_classes=2, label="Prediksi"),
    title="Skin Disease Classifier (6 kelas)",
    flagging_mode="never",
).launch(share=True)  # share=True -> membuat link publik